# IMPORT LIBRARY

In [ ]:
import datetime
import mysql.connector
from mysql.connector import Error
from mysql.connector import errorcode
import time

from pytz import timezone

import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date

import importlib, importlib.util

import warnings
warnings.filterwarnings("ignore")

from sqlalchemy import create_engine
import psycopg2

import requests
import json

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec

import pymysql
pd.set_option('display.max_columns', None)

# CONFIG DB

In [ ]:
# Import Config Data
import importlib, importlib.util

def module_from_file(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

config_dl_dwh = module_from_file("config_dw", "./db_config/config_db_datawarehouse.py")
config_jiwaplus = module_from_file("config_dw", "./db_config/config_db_jiwaplus.py")
config_ksj_link = module_from_file("config_dw", "./db_config/config_db_ksj_link.py")

In [ ]:
def dl_dwh_db_conn(db_name):
    return pymysql.connect(
        host=config_dl_dwh.hostname,
        db=db_name,
        user=config_dl_dwh.username,
        password=config_dl_dwh.password,
        port=3306
    )

In [ ]:
def jiwa_db_conn():
    return psycopg2.connect(
        host=config_jiwaplus.hostname,
        database=config_jiwaplus.database,
        user=config_jiwaplus.username,
        password=config_jiwaplus.password,
        port='5432'
    )

In [ ]:
def ksj_link_db_conn(db_name):
    return psycopg2.connect(
        host=config_ksj_link.hostname,
        database=db_name,
        user=config_ksj_link.username,
        password=config_ksj_link.password,
        port='5432'
    )

# FUNCTION

In [ ]:
def get_rider_hourly_transactions(start_date_str, end_date_str):
    connection = ksj_link_db_conn(config_ksj_link.d_transaction)
    cursor = connection.cursor()
    
    query_1 = """
        SELECT 
            t.rider_code,
            t.transaction_date AS date,

            EXTRACT(
                WEEK FROM 
                COALESCE(
                    t.transaction_datetime_utc,
                    t.transaction_date::timestamp
                )
            )::INT AS week,

            EXTRACT(
                HOUR FROM transaction_datetime_utc + INTERVAL '7 hours'
            )::INT AS hour,

            t.payment_type,

            COUNT(DISTINCT t.id) AS total_orders,
            SUM(tp.qty) AS total_qty,
            SUM(tp.total_price_after_discount) AS total_sales

        FROM transactions t

        JOIN transaction_products tp
            ON t.id = tp.transaction_id

        WHERE
            t.transaction_status = 'COMPLETED'
            AND t.rider_code != 'JW000065'
            AND t.transaction_date BETWEEN '{0}' AND '{1}'

        GROUP BY 1,2,3,4,5
    """.format(start_date_str, end_date_str)
    
    cursor.execute(query_1)

    record = cursor.fetchall()
    columns = [col[0] for col in cursor.description]
    df = pd.DataFrame(record, columns=columns)

    cursor.close()
    connection.close()
    
    return df

In [ ]:
def get_rider_timeslot_tito(start_date_str, end_date_str):
    connection = ksj_link_db_conn(config_ksj_link.d_jiwa_ksj)
    cursor = connection.cursor()
    
    query_1 = """
        WITH rider_shift AS (
            SELECT 
                rider_code,
                date,
                checkin + INTERVAL '7 hours' AS checkin,
                checkout + INTERVAL '7 hours' AS checkout
            FROM rider_transfer_log

            WHERE checkin IS NOT NULL
                AND checkout IS NOT NULL
                AND date BETWEEN '{0}' AND '{1}'
        )

        SELECT
            rider_code,
            date,
            checkin,
            checkout,
            TO_CHAR(timeslot, 'HH24') AS hour_slot

        FROM rider_shift

        CROSS JOIN LATERAL
            generate_series(
                date_trunc('hour', checkin),
                date_trunc('hour', checkout),
                interval '1 hour'
            ) AS timeslot

        ORDER BY date, rider_code, hour_slot;
    """.format(start_date_str, end_date_str)
    
    cursor.execute(query_1)

    record = cursor.fetchall()
    columns = [col[0] for col in cursor.description]
    df = pd.DataFrame(record, columns=columns)

    cursor.close()
    connection.close()
    
    return df

In [ ]:
def get_rider_mapping():
    connection = dl_dwh_db_conn(config_dl_dwh.db_resource)
    cursor = connection.cursor()
    
    query_1 = f"""
        SELECT DISTINCT
            drm.rider_id,
            drm.jilid,
            drm.hub_name,
            domc.region,
            domc.provinsi,
            domc.kota,
            domc.sejuta_jiwa_type
        FROM sejutajiwa_riders_hubs_mapping drm
        JOIN outlet_mapping_custom domc ON drm.jilid = domc.jilid
    """
    
    cursor.execute(query_1)

    record = cursor.fetchall()
    columns = [col[0] for col in cursor.description]
    df = pd.DataFrame(record, columns=columns)

    cursor.close()
    connection.close()
    
    return df

In [ ]:
load_data_by = 'lahiardhan@jiwagroup.com'

---

# EXTRACT

## Get Date

In [ ]:
yesterday = datetime.today().date() - timedelta(days=1)

# start_date = today
# end_date = start_date - timedelta(days=1)

start_date_str = yesterday.strftime('%Y-%m-%d')
end_date_str = yesterday.strftime('%Y-%m-%d')

# start_date_str = '2026-05-01'
# end_date_str = '2026-05-21'

print("today     :", yesterday)
print("start_day :", start_date_str)
print("end_day   :", end_date_str)

In [ ]:
df_hourly_transactions = get_rider_hourly_transactions(start_date_str, end_date_str)

print(len(df_hourly_transactions))
df_hourly_transactions.head(3)

In [ ]:
df_rider_timeslot = get_rider_timeslot_tito(start_date_str, end_date_str)

df_rider_timeslot.head(3)

In [ ]:
df_rider_mapping = get_rider_mapping()

df_rider_mapping.tail(3)

# MAIN PROGRESS

## 2.1 Eliminate SJO Riders
Because SJO riders didnt do TI-TO 

In [ ]:
list_rider_hub = df_rider_mapping[df_rider_mapping['sejuta_jiwa_type'] == 'HUB']['rider_id'].unique().tolist()

df_transactions_filtered = df_hourly_transactions[df_hourly_transactions['rider_code'].isin(list_rider_hub)]

print(len(df_transactions_filtered))
df_transactions_filtered.head(3)

## 2.2 Cleaning Data

In [ ]:
# Samakan tipe data date
df_transactions_filtered['date'] = pd.to_datetime(df_transactions_filtered['date'])
df_rider_timeslot['date'] = pd.to_datetime(df_rider_timeslot['date'])

# Samakan tipe data hour
df_transactions_filtered['hour'] = df_transactions_filtered['hour'].astype(int)
df_rider_timeslot['hour_slot'] = df_rider_timeslot['hour_slot'].astype(int)

# Outer merge
df_merged = pd.merge(
    df_transactions_filtered,
    df_rider_timeslot,
    left_on=['rider_code', 'date', 'hour'],
    right_on=['rider_code', 'date', 'hour_slot'],
    how='outer'
)

df_merged

In [ ]:
# Cari min & max hour_slot per rider per tanggal
df_merged['slot_min'] = df_merged.groupby(['rider_code', 'date'])['hour_slot'].transform('min')
df_merged['slot_max'] = df_merged.groupby(['rider_code', 'date'])['hour_slot'].transform('max')

# Flag apakah rider+date punya slot dan/atau transaksi
df_merged['has_slot'] = df_merged.groupby(['rider_code', 'date'])['hour_slot'].transform(lambda x: x.notna().any())
df_merged['has_trx']  = df_merged.groupby(['rider_code', 'date'])['hour'].transform(lambda x: x.notna().any())

In [ ]:
# CASE 1: Ada transaksi tapi jam transaksinya di luar rentang hour_slot
case1 = df_merged[
    df_merged['has_slot'] & 
    df_merged['has_trx'] & 
    df_merged['hour'].notna() & 
    ((df_merged['hour'] < df_merged['slot_min']) | (df_merged['hour'] > df_merged['slot_max']))
][['rider_code', 'date', 'hour', 'slot_min', 'slot_max', 'total_orders']].drop_duplicates()

# CASE 2: Ada transaksi tapi rider+tanggal itu TIDAK PUNYA hour_slot sama sekali
case2 = df_merged[
    ~df_merged['has_slot'] & 
    df_merged['hour'].notna()
][['rider_code', 'date', 'hour', 'total_orders', 'total_sales']].drop_duplicates()

# CASE 3: Ada hour_slot tapi rider+tanggal itu TIDAK ADA transaksi sama sekali
case3 = df_merged[
    ~df_merged['has_trx'] & 
    df_merged['hour_slot'].notna()
][['rider_code', 'date', 'total_orders', 'slot_min', 'slot_max']].drop_duplicates()

In [ ]:
print("=" * 50)
print("ASSESSMENT SUMMARY")
print("=" * 50)
print(f"CASE 1 - Transaksi di luar rentang jam kerja : {case1[['rider_code','date']].drop_duplicates().shape[0]} rider-date ({len(case1)} baris)")
print(f"CASE 2 - Ada transaksi, gapunya hour_slot    : {case2[['rider_code','date']].drop_duplicates().shape[0]} rider-date ({len(case2)} baris)")
print(f"CASE 3 - Punya hour_slot, tapi gaada transaksi: {case3[['rider_code','date']].drop_duplicates().shape[0]} rider-date ({len(case3)} baris)")
print()

print("--- CASE 1 Detail ---")
display(case1.sort_values(['rider_code', 'date', 'hour']))

print("--- CASE 2 Detail ---")
display(case2.sort_values(['rider_code', 'date', 'hour']))

print("--- CASE 3 Detail ---")
display(case3.sort_values(['rider_code', 'date']))

## 2.3 Handle 3 Cases

### a. Case 1 & Exclude Case 3

In [ ]:
import numpy as np

# ─── a. Case 1: Snap out-of-range hours ──────────────────────────────────────
df_slot_range = df_rider_timeslot.groupby(['rider_code', 'date']).agg(
    slot_min=('hour_slot', 'min'),
    slot_max=('hour_slot', 'max')
).reset_index()

df_trx = df_transactions_filtered.merge(df_slot_range, on=['rider_code', 'date'], how='left')

df_trx['hour'] = np.where(
    df_trx['slot_min'].notna() & (df_trx['hour'] < df_trx['slot_min']),
    df_trx['slot_min'],
    np.where(
        df_trx['slot_max'].notna() & (df_trx['hour'] > df_trx['slot_max']),
        df_trx['slot_max'],
        df_trx['hour']
    )
).astype(int)

df_trx_fixed = (
    df_trx
    .groupby(['rider_code', 'date', 'week', 'hour', 'payment_type'])
    .agg(
        total_orders=('total_orders', 'sum'),
        total_qty=('total_qty', 'sum'),
        total_sales=('total_sales', 'sum'),
    )
    .reset_index()
)

# ─── b. Case 2: Populate slots dari range transaksi ──────────────────────────
case2_rider_dates = (
    df_trx_fixed
    .groupby(['rider_code', 'date'])['hour']
    .agg(hour_min='min', hour_max='max')
    .reset_index()
    .merge(df_slot_range[['rider_code', 'date']], on=['rider_code', 'date'], how='left', indicator=True)
    .query('_merge == "left_only"')
    .drop(columns='_merge')
)

case2_slots_list = []
for _, row in case2_rider_dates.iterrows():
    for h in range(int(row['hour_min']), int(row['hour_max']) + 1):
        case2_slots_list.append({
            'rider_code': row['rider_code'],
            'date': row['date'],
            'hour_slot': h
        })

if case2_slots_list:
    df_case2_slots = pd.DataFrame(case2_slots_list)
    df_case2_slots['date'] = pd.to_datetime(df_case2_slots['date'])
    df_timeslot_fixed = pd.concat([df_rider_timeslot, df_case2_slots], ignore_index=True).drop_duplicates()
else:
    print("Case 2: Tidak ada rider yang masuk case ini, semua rider sudah punya slot data.")
    df_timeslot_fixed = df_rider_timeslot.copy()

# ─── c. Case 3: Exclude rider+date yang ZERO transaksi sepanjang hari ────────
rider_dates_with_trx = df_trx_fixed[['rider_code', 'date']].drop_duplicates()

df_timeslot_base = df_timeslot_fixed.merge(
    rider_dates_with_trx,
    on=['rider_code', 'date'],
    how='inner'  # hanya rider+date yang minimal punya 1 transaksi di hari itu
)

# ─── d. Final merge: hour_slot sebagai BASE, transaksi di-join ke kanan ──────
df_final = df_timeslot_base.merge(
    df_trx_fixed,
    left_on=['rider_code', 'date', 'hour_slot'],
    right_on=['rider_code', 'date', 'hour'],
    how='left'
)

print(len(df_final))
df_final.head(10)

## Merge Final Dataframe

In [ ]:
# Fill NULL week column with respective week
df_final['week'] = df_final['week'].fillna(
    pd.to_datetime(df_final['date']).dt.isocalendar().week.astype(int)
)

df_final_merged = df_final.merge(
    df_rider_mapping,
    left_on='rider_code',
    right_on='rider_id',
    how='left'
).drop(columns=['rider_code'])

# Reorder columns
df_final_merged = df_final_merged[[
    'rider_id', 'jilid', 'hub_name', 'region', 'provinsi', 'kota', 'sejuta_jiwa_type',
    'date', 'checkin', 'checkout', 'hour_slot', 'week', 'hour',
    'payment_type', 'total_orders', 'total_qty', 'total_sales'
]]


# Strip whitespace strings then replace with NULL
df_final_merged = df_final_merged.apply(lambda col: col.map(lambda x: x.strip() if isinstance(x, str) else x))
df_final_merged = df_final_merged.replace({'': np.nan, 'None': np.nan, 'nan': np.nan, 'null': np.nan, 'NULL': np.nan})

df_final_merged['load_data_by'] = load_data_by 

print(len(df_final_merged))
df_final_merged

In [ ]:
df_cek_nominal = df_hourly_transactions.merge(df_rider_mapping[['rider_id', 'sejuta_jiwa_type']], left_on='rider_code', right_on='rider_id', how='left')

print(df_cek_nominal[df_cek_nominal['sejuta_jiwa_type'] == 'HUB']['total_sales'].sum())
print(df_final_merged['total_sales'].sum())

# DUMP

In [ ]:
engine = create_engine("mysql+pymysql://{user}:{pw}@{hostsname}/{db}".format(
    hostsname=config_dl_dwh.hostname,
    user=config_dl_dwh.username,
    pw=config_dl_dwh.password,
    db=config_dl_dwh.db_target
))

In [ ]:
import math

batch_size = 50_000
total_rows = len(df_final_merged)
total_batches = math.ceil(total_rows / batch_size)

for i in range(total_batches):
    start = i * batch_size
    end = start + batch_size
    batch = df_final_merged.iloc[start:end]
    
    batch.to_sql(
        name="hourly_sejutajiwa_rider_performance",
        con=engine,
        if_exists="append",
        index=False,
        chunksize=50_000,
        method="multi"
    )
    
    print(f"Batch {i+1}/{total_batches} inserted ({end if end < total_rows else total_rows}/{total_rows} rows)")